# Clipper Whisper Worker (Colab GPU)
Jalankan semua cell berurutan. Di cell terakhir, copy URL tunnel + token ke Settings Clipper.


In [ ]:
!pip install -q faster-whisper fastapi uvicorn python-multipart
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared


In [ ]:
import os
os.environ['WORKER_TOKEN'] = 'ganti-dengan-token-rahasia'
os.environ['WHISPER_MODEL'] = 'large-v3-turbo'
os.environ['WHISPER_COMPUTE'] = 'float16'


In [ ]:
!wget -q https://raw.githubusercontent.com/Maulana2505/clipper-Remotion-renderer-/main/colab/worker.py -O /content/worker.py
import os
print('worker.py ready:', os.path.exists('/content/worker.py'))


In [ ]:
import subprocess, time, re, threading

def run_worker():
    subprocess.run(['uvicorn','worker:app','--host','0.0.0.0','--port','8010'])
threading.Thread(target=run_worker, daemon=True).start()
time.sleep(10)

proc = subprocess.Popen(['cloudflared','tunnel','--url','http://127.0.0.1:8010'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in proc.stdout:
    import re
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
print('TUNNEL URL:', url)
print('TOKEN:', os.environ['WORKER_TOKEN'])
proc.wait()
